In [4]:
import json
import pandas as pd
from elasticsearch import Elasticsearch, helpers


client = Elasticsearch(
  "https://localhost:9200/",
  api_key="QktuNGlKTUJyZmtvclV0N2tHSFA6Q0FRYmExLVhTemkySGpvalBFMndDUQ==",
  verify_certs=False
)

c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\elasticsearch\_sync\client\__init__.py:402: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [86]:
def create_document(index: str, item_id:int, page_id:int, text:str, title:str) -> dict:
    return {
            "_index": index,
            "_id": f"_{str(item_id)}",  # Unique ID for each document
            "_source": {
                "item_id": int(item_id),
                "page_id": page_id,
                "text": str(text),
                "title": str(title)
            }
        }

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [91]:
def process_data_in_batches(client, index, jsonl_file_path, page_data, batch_size):
    batch_data, failed_pages,not_processed = [],[],[]
    batch_number = 0
    with open(jsonl_file_path, 'r', encoding='utf-8') as jsonl_file:
        for _, line in enumerate(jsonl_file):
            try:
                # JSON-Zeile parsen und zur Batch-Liste hinzufügen
                text = ""
                line = json.loads(line.strip())
                for section in line['sections']:
                    if section['name'] not in ["See also", "External links"]:
                        text += section['text'].replace('"',"")+ "\n"
                page_id = line["page_id"]
                page = page_data[page_data["page_id"]==page_id]
                item_id = page.item_id.iloc[0]
                title = page.title.iloc[0]
                batch_data.append(create_document(index, item_id, page_id, text, title))
                try:
                    if len(batch_data) % batch_size == 0:
                        batch_number += 1
                        helpers.bulk(client, batch_data)
                        batch_data = []
                        if batch_number % 10 == 0:
                            print("Batch number in %: ", batch_number)
                except Exception as e:
                    print(e)
                    print("Error in batch number uploading: ", batch_number)
                    not_processed.append(batch_data)
                    continue
            except Exception as e:
                print(e)
                print("Error in item: ", item_id, "Error in line: ", line)
                failed_pages.append(page_id)
                continue
    return batch_number, failed_pages, not_processed

In [7]:
page_data = pd.read_csv("../raw_data/page.csv")
jsonl_file_path = '../raw_data/link_annotated_text.jsonl'

In [92]:
test, failed_pages, not_processed = process_data_in_batches(client, "wikidata", jsonl_file_path, page_data, 1000)

Batch number in %:  10
Batch number in %:  20
Batch number in %:  30
Batch number in %:  40
Batch number in %:  50
Batch number in %:  60
Batch number in %:  70
Batch number in %:  80
Batch number in %:  90
Batch number in %:  100
Batch number in %:  110
Batch number in %:  120
Batch number in %:  130
Batch number in %:  140
Batch number in %:  150
Batch number in %:  160
Batch number in %:  170
Batch number in %:  180
Batch number in %:  190
Batch number in %:  200
Batch number in %:  210
Batch number in %:  220
Batch number in %:  230
Batch number in %:  240
Batch number in %:  250
Batch number in %:  260
Batch number in %:  270
Batch number in %:  280
Batch number in %:  290
Batch number in %:  300
Batch number in %:  310
Batch number in %:  320
Batch number in %:  330
Batch number in %:  340
Batch number in %:  350
Batch number in %:  360
Batch number in %:  370
Batch number in %:  380
Batch number in %:  390
Batch number in %:  400
Batch number in %:  410
Batch number in %:  420
B

In [95]:
# DIE LETZTEN X fehlen noch!!!

In [94]:
not_processed

[]

In [36]:
process_data_in_batches(client, "wikidata", jsonl_file_path, page_data, 1)

1 document(s) failed to index.
Error in batch number uploading:  1
1 document(s) failed to index.
Error in batch number uploading:  2
1 document(s) failed to index.
Error in batch number uploading:  3
1 document(s) failed to index.
Error in batch number uploading:  4
1 document(s) failed to index.
Error in batch number uploading:  5
1 document(s) failed to index.
Error in batch number uploading:  6
1 document(s) failed to index.
Error in batch number uploading:  7
1 document(s) failed to index.
Error in batch number uploading:  8
1 document(s) failed to index.
Error in batch number uploading:  9
1 document(s) failed to index.
Error in batch number uploading:  10
1 document(s) failed to index.
Error in batch number uploading:  11
1 document(s) failed to index.
Error in batch number uploading:  12
1 document(s) failed to index.
Error in batch number uploading:  13
1 document(s) failed to index.
Error in batch number uploading:  14
1 document(s) failed to index.
Error in batch number uplo

KeyboardInterrupt: 

In [12]:
test = {'page_id': 53972, 'sections': [{'name': 'Introduction', 'text': 'Lorenz Milton Hart (May 2, 1895 – November 22, 1943) was the lyricist half of the Broadway songwriting team Rodgers and Hart. Some of his more famous lyrics include "Blue Moon," "Mountain Greenery," "The Lady Is a Tramp," "Manhattan," "Where or When," "Bewitched, Bothered and Bewildered," "Falling in Love with Love," "Have You Met Miss Jones?," "My Funny Valentine," "I Could Write a Book", "This Can\'t Be Love", "With a Song in My Heart", "It Never Entered My Mind", and "Isn\'t It Romantic?".', 'link_lengths': [8, 8, 11, 16, 9, 17, 19, 9, 13, 34, 25, 24, 18, 20, 18, 23, 24, 18], 'link_offsets': [61, 82, 91, 108, 166, 179, 200, 223, 236, 253, 291, 320, 348, 370, 394, 416, 443, 475], 'target_page_ids': [411266, 725252, 94154, 199824, 1826002, 12527103, 6483476, 7618310, 5864130, 355627, 7806330, 7810800, 2021686, 12527223, 6314068, 3898268, 12576670, 7806522]}, {'name': 'Life and career', 'text': 'Hart was born in Harlem, New York City, the elder of two sons, to Jewish immigrant parents, Max M. and Frieda (Isenberg) Hart, of German background. Through his mother, he was a great-grandnephew of the German poet Heinrich Heine. His father, a business promoter, sent Hart and his brother to private schools. (His brother, Teddy Hart, also went into theatre and became a musical comedy star. Teddy Hart\'s wife, Dorothy Hart, wrote a biography of Lorenz Hart.) Hart received his early education from Columbia Grammar School and then attended Columbia University School of Journalism for two years. In 1919 a friend introduced him to Richard Rodgers, and the two joined forces to write songs for a series of amateur and student productions. By 1918, Hart was working for the Shubert brothers, partners in theatre, translating German plays into English. In 1919, his and Rodgers\' song "Any Old Place With You" was included in the Broadway musical comedy A Lonely Romeo. In 1920, six of their songs were used in the musical comedy Poor Little Ritz Girl, which also had music by Sigmund Romberg. They were hired to write the score for the 1925 Theatre Guild production The Garrick Gaieties, the success of which brought them acclaim. Rodgers and Hart subsequently wrote the music and lyrics for 26 Broadway musicals during a more-than-20-year partnership that ended shortly before Hart\'s early death. Their "big four" were Babes in Arms, The Boys From Syracuse, Pal Joey, and On Your Toes. The Rodgers and Hart songs have been described as intimate and destined for long lives outside the theater. Many of their songs are standard repertoire for singers and jazz instrumentalists. Notable singers who have performed and recorded their songs have included Frank Sinatra, Doris Day, Billie Holiday, Ella Fitzgerald, Blossom Dearie, and Carly Simon. Hart has been called "the expressive bard of the urban generation which matured during the interwar years." But the "encomiums suggest(ing) that Larry Hart was a poet" caused his friend and fellow writer Henry Myers to state otherwise. "Larry in particular was primarily a showman. If you can manage to examine his songs technically, and for the moment elude their spell, you will see that they are all meant to be acted, that they are part of a play. Larry was a playwright." Rodgers and Hart wrote music and lyrics for several films, including Love Me Tonight (1932), The Phantom President (1932), Hallelujah, I\'m a Bum (1933), and Mississippi (1935). With their successes, during the Great Depression Hart was earning $60,000 annually, and he became a magnet for many people. He gave numerous large parties. Beginning in 1938, he traveled more often and suffered from his drinking. Nevertheless, Rodgers and Hart continued working together through mid-1942, with their final new musical being 1942\'s By Jupiter. The New York Times reported on July 23, 1942: "The Theatre Guild announced yesterday that Richard Rodgers, Lorenz Hart and Oscar Hammerstein II will soon begin work on a musical version of Lynn Riggs\'s folk-play, \'Green Grow the Lilacs,\' which the Guild produced for sixty-four performances at the Guild Theatre in 1931." The musical opened March 31, 1943 as Oklahoma! but Hart had exited, leaving Rodgers and his new partner Hammerstein as the composer and lyricist. Hart, meanwhile, was much affected by his mother\'s death in late April 1943. Regrouping somewhat, Rodgers and Hart teamed a final time in the fall of 1943 for a revival of A Connecticut Yankee. Six new numbers, including "To Keep My Love Alive", were written for this reworked version of the play; it would prove to be Hart\'s last lyric. Hart had taken off the night of the opening and was gone for two days. He was found ill in a hotel room and taken to Doctors Hospital, Upper East Side, but died within a few days. After Hart\'s death, Rodgers continued his collaboration with Oscar Hammerstein II. Theirs was a long and successful collaboration, one which made them one of the most successful composing teams of the 20th century.', 'link_lengths': [6, 13, 6, 9, 14, 23, 19, 15, 7, 21, 15, 13, 20, 13, 22, 8, 12, 13, 9, 14, 15, 14, 11, 15, 21, 21, 11, 16, 10, 18, 13, 20, 9, 20, 21, 16, 15], 'link_offsets': [17, 25, 66, 73, 215, 500, 542, 633, 774, 1028, 1075, 1140, 1165, 1419, 1434, 1458, 1472, 1751, 1766, 1777, 1793, 1810, 1830, 2389, 2413, 2443, 2477, 2530, 2846, 2858, 2909, 2981, 3217, 3498, 3548, 3781, 3799], 'target_page_ids': [54861, 645042, 25955086, 4599312, 104641, 431754, 6310, 52274, 1881890, 48979994, 400166, 2247269, 1029734, 145622, 935901, 5925685, 1910356, 11181, 8300, 50420, 50350, 1641899, 177233, 74913, 10394188, 7250130, 8288693, 19283335, 9234542, 30680, 2247269, 22753, 77420, 2206538, 12576698, 28525689, 327270]}, {'name': 'Musical style', 'text': 'According to Thomas Hischak, Hart "had a remarkable talent for polysyllabic and internal rhymes", and his lyrics have often been praised for their wit and technical sophistication. According to Stephen Holden, a writer in The New York Times, "Many of Hart\'s ballad lyrics conveyed a heart-stopping sadness that reflected his conviction that he was physically too unattractive to be lovable." The New York Times writer also noted that "In his lyrics, as in his life, Hart stands as a compellingly lonely figure. Although he wrote dozens of songs that are playful, funny and filled with clever wordplay, it is the rueful vulnerability beneath their surface that lends them a singular poignancy."', 'link_lengths': [14, 18], 'link_offsets': [80, 222], 'target_page_ids': [583616, 30680]}, {'name': 'Personal life and death', 'text': 'Hart lived with his widowed mother. He suffered from alcoholism, and would sometimes disappear for weeks at a time on alcoholic binges. Holden writes: Hart suffered from depression throughout his life. His erratic behavior was often the cause of friction between him and Rodgers and led to a breakup of their partnership in 1943 before his death. Rodgers then began collaborating with Oscar Hammerstein II. Devastated by the death of his mother seven months earlier, Hart died in New York City of pneumonia from exposure on November 22, 1943, after drinking heavily. His remains are buried in Mount Zion Cemetery in Queens County, New York. The circumstances of his life were heavily edited and romanticized for the 1948 MGM biopic Words and Music.', 'link_lengths': [10, 10, 20, 9, 19, 13, 8, 3, 15], 'link_offsets': [53, 170, 385, 497, 593, 616, 631, 721, 732], 'target_page_ids': [2965, 840273, 22753, 52135, 23076700, 45579, 8210131, 58819, 3923707]}, {'name': 'Selected stage works', 'text': "* 1920 Poor Little Ritz Girl * 1925 The Garrick Gaieties * 1927 A Connecticut Yankee, based on the Mark Twain novel * 1928 Present Arms * 1935 Jumbo * 1936 On Your Toes * 1937 Babes in Arms * 1938 The Boys from Syracuse, based on William Shakespeare's The Comedy of Errors * 1938 I Married an Angel * 1938 Too Many Girls * 1940 Higher and Higher * 1940 Pal Joey, based on John O'Hara's work * 1942 By Jupiter", 'link_lengths': [21, 20, 20, 10, 12, 5, 12, 13, 22, 19, 20, 18, 14, 17, 8, 11, 10], 'link_offsets': [7, 36, 64, 99, 123, 143, 156, 176, 197, 230, 252, 280, 306, 328, 353, 372, 398], 'target_page_ids': [48979994, 1029734, 2206538, 154450, 9781153, 1088182, 1910356, 145622, 935901, 32897, 324904, 934376, 9846719, 9847084, 5925685, 209294, 9234542]}, {'name': 'Notable songs', 'text': '* "A Ship Without a Sail" * "Bewitched, Bothered and Bewildered" * "Blue Moon" * "Blue Room" * "Dancing on the Ceiling" * "Falling in Love with Love" * "Glad to Be Unhappy" * "Have You Met Miss Jones?" * "He Was Too Good to Me" * "I Could Write a Book" * "I Didn\'t Know What Time It Was" * "I Wish I Were in Love Again" * "I\'ll Tell The Man In The Street" * "I\'ve Got Five Dollars" * "Isn\'t It Romantic?" * "It Never Entered My Mind" * "It\'s Easy to Remember" * "Johnny One Note" * "Little Girl Blue" * "Lover" * "Manhattan" * "Mountain Greenery" * "My Funny Valentine" * "My Heart Stood Still" * "My Romance" * "Sing for Your Supper" * "Spring Is Here" * "Ten Cents a Dance" * "The Lady Is a Tramp" * "The Most Beautiful Girl in the World" * "There\'s a Small Hotel" * "This Can\'t Be Love" * "Thou Swell" * "To Keep My Love Alive" * "Where or When" * "With a Song in My Heart" * "You Took Advantage of Me"', 'link_lengths': [21, 34, 9, 9, 22, 25, 18, 24, 21, 20, 30, 27, 31, 21, 18, 24, 21, 15, 16, 5, 9, 17, 18, 20, 10, 20, 14, 17, 19, 36, 21, 18, 10, 21, 13, 23, 24], 'link_offsets': [3, 29, 68, 82, 96, 123, 153, 176, 205, 231, 256, 291, 323, 359, 385, 408, 437, 463, 483, 504, 514, 528, 550, 573, 598, 613, 638, 657, 679, 703, 744, 770, 793, 808, 834, 852, 880], 'target_page_ids': [52274, 355627, 1826002, 2001998, 12576731, 7806330, 12844974, 7810800, 9923197, 12527223, 6498740, 12362345, 19897042, 12527012, 7806522, 12576670, 6706698, 12532352, 7617900, 1950276, 7618310, 12527103, 2021686, 12527075, 3430060, 12154563, 12532226, 3292592, 6483476, 6276833, 12576468, 6314068, 10542365, 12576698, 5864130, 3898268, 12527701]}, {'name': 'Further reading', 'text': "*Friends of the USC Libraries. The Hart of the Matter: A Celebration of Lorenz Hart, September 30, 1973. [Los Angeles]: Friends of the USC Libraries, University of Southern California, 1973. *Hart, Dorothy. Thou Swell, Thou Witty: The Life and Lyrics of Lorenz Hart, New York: Harper & Row, 1976. *Marmorstein, Gary. A Ship Without A Sail: The Life of Lorenz Hart, New York: Simon & Schuster, 2012. *Marx, Samuel, and Jan Clayton. Rodgers & Hart: Bewitched, Bothered, and Bedeviled: An Anecdotal Account, New York: Putnam, 1976. *Nolan, Frederick W. Lorenz Hart: A Poet on Broadway. New York: Oxford University Press, 1994. *Furia, Philip. The Poets of Tin Pan Alley: A History of America's Great Lyricists. New York: Oxford University Press, 1990.", 'link_lengths': [12, 16, 19, 23, 13, 23], 'link_offsets': [277, 375, 530, 593, 625, 718], 'target_page_ids': [1273102, 1001882, 1581819, 48518, 15994387, 48518]}]}


In [13]:
text = ""
line = test
for section in line['sections']:
    if section['name'] not in ["See also", "External links"]:
        text += section['text'] + " "

        page_id = line["page_id"]
        page = page_data[page_data["page_id"]==page_id]
        item_id = page.item_id.iloc[0]
        title = page.title.iloc[0]
print(create_document("wikidata", item_id, page_id, text, title))

{'_index': 'wikidata', '_id': '_725828', '_source': {'item_id': 725828, 'page_id': 53972, 'text': 'Lorenz Milton Hart (May 2, 1895 – November 22, 1943) was the lyricist half of the Broadway songwriting team Rodgers and Hart. Some of his more famous lyrics include "Blue Moon," "Mountain Greenery," "The Lady Is a Tramp," "Manhattan," "Where or When," "Bewitched, Bothered and Bewildered," "Falling in Love with Love," "Have You Met Miss Jones?," "My Funny Valentine," "I Could Write a Book", "This Can\'t Be Love", "With a Song in My Heart", "It Never Entered My Mind", and "Isn\'t It Romantic?". Hart was born in Harlem, New York City, the elder of two sons, to Jewish immigrant parents, Max M. and Frieda (Isenberg) Hart, of German background. Through his mother, he was a great-grandnephew of the German poet Heinrich Heine. His father, a business promoter, sent Hart and his brother to private schools. (His brother, Teddy Hart, also went into theatre and became a musical comedy star. Teddy Hart